## ADIP Phase 2: Data Standardization Investigation

### Objective
Investigate raw Parquet data from Phase 1 ingestion to identify schema inconsistencies, missing values, type mismatches, and structural anomalies before enforcing `StandardizedRecord` validation.

### Investigation Targets
- **API ingestion output**: `api_auth*.csv`, `api_ingest*.csv`
- **Scraper output**: `scraper*.csv`

### Key Questions
1. Do all files share identical column names and types?
2. Are timestamps consistently formatted (UTC, timezone-aware)?
3. What percentage of records fail `StandardizedRecord` validation?
4. Which fields require coercion (string → float, object → datetime)?
5. What edge cases exist (empty strings, NaN patterns, outliers)?

### BEGIN!

  **FRMAEWORK FOR PHASE 1 - API_INGEST.CSV** 
--------------
#### 1. Phase Overview
In this phase, we ingest the raw Ecommerce Sales dataset and conduct a high-level inspection.
Goals:

- Load the dataset in a reproducible way.

- Examine basic metadata and structural integrity.

- Identify potential data quality issues for later cleaning.

- Document first impressions and possible red flags.

In [ ]:
!pip install pandas
!pip install numpy

In [ ]:
import pandas as pd
import numpy as np

: 

In [2]:
# Check the current working directory
import os
os.getcwd()

'c:\\Users\\CHARLES\\Desktop\\ADIP-Intelligence-lab\\eda_notebook'

In [ ]:
df = pd.read_csv("../.data_cache/api_ingest.csv")  

#### Basic Structural Inspection
Checklist:

- Shape (.shape)

- Column names (.columns)

- Data info: data types & nullcouns  (.info)

- Sample rows (.head() / .tail())

In [ ]:
df.head()

In [ ]:
print(f"shape :{df.shape}") 

**Dropping deal_price, final_price & description columns - These are ghost columns with 0 data points.**

In [ ]:
df.drop(columns=['deal_price', 'final_price', 'description'], inplace=True)
df.columns

In [ ]:
df.info()

#### Inspection For Missing & Null Value  
Checklist:

- Count missing values per column (.isnull().sum()).

- Percentage of missing values.

- Repopulate missing data (e.g., 'NA', '?', 'n/a').



In [ ]:
df.isna().sum()

In [ ]:
percent_missing = df.isna().sum() * 100 / len(df)
print(percent_missing)

***Upon observation - Over half of the brand rows consist of missing value. This needs to be addressed!***

In [ ]:
# Extract the brands from the names column by means of split function, and assign it to the empty brand column. 
names = df["name"].str.split().str[0]
if df["brand"].isna().any() or (df["brand"] == "").any():
    df["brand"] = names 

In [ ]:
df["brand"].isna().sum() 

In [ ]:
df["brand"].value_counts() 

In [ ]:
# Data cleaning: Replace the incorrect brand name "JCO4" with the correct brand name "HP".

if (df["brand"] == "JCO4").any():
    df["brand"] = df["brand"].replace("JCO4", "HP")
df["brand"] 

In [ ]:
df.head()

***Unpacking the nested strings stored within the following columns - seller, stock & product_rating*** 

In [ ]:
import ast

In [ ]:
def unpack_dict(val, target_key, default_val=None):
    """
    Safely turns a string like "{'in_stock': True, 'quantity': 20}" into a dictionary,
    and extracts the target_key.
    """
    if pd.isna(val):
        return default_val
        
    try:
        # Translate string -> dict
        parsed_dict = ast.literal_eval(str(val))
        
        # Grab the key we want safely
        return parsed_dict.get(target_key, default_val)
        
    except (ValueError, SyntaxError):
        # If the string is broken, don't crash the notebook, just return the default
        return default_val

print("✅ Unpacker tool loaded into memory.")

In [ ]:
# 1. Extract the quantity from the stock column
df['stock'] = df['stock'].apply(lambda x: unpack_dict(x, target_key='quantity', default_val=0))

# 2. Extract the name from the seller column
df['seller'] = df['seller'].apply(lambda x: unpack_dict(x, target_key='name', default_val='Unknown'))

# 3. Extract the average quality rating from the product_rating column
df["product_rating"] = df["product_rating"].apply(lambda x: ast.literal_eval(str(x)) if pd.notna(x) else {})
df["product_rating"] = df["product_rating"].apply(lambda d: d.get('quality', {}).get("average", np.nan)) 

# 4. VISUALIZE THE RESULTS
df.head()

In [ ]:
df.head()

,name,brand,price,image_thumbnail,sku,seller,stock,product_rating,fetched_at
0,JCO4 LAPTOP BATTERY,HP,24725,/T/I/118566_1604505433.jpg,5000867,Konga,1,0.0,2025-11-11 19:49:08
1,Basilisk Gaming Mouse Classic Black,Basilisk,244000,/R/U/57289_1727279653.jpg,6566278,YOMILINCON BRAND,20,0.0,2025-11-11 19:49:08
2,Ht03xl In-built Battery,Ht03xl,35075,/S/H/118566_1618241664.jpg,5228829,Konga,2,0.0,2025-11-11 19:49:08
3,HP Big-Mouth Laptop Charger - 18.5V 3.5A,HP,10000,/H/P/HP-Big-Mouth-Laptop-Charger---18-5V-3-5A-...,2353031,YOMILINCON BRAND,9,4.5,2025-11-11 19:49:08
4,Baseus Ultrajoy 5w1 5-port Hub docking station...,Baseus,49000,/F/Z/57289_1760001914.jpg,6861874,YOMILINCON BRAND,2,0.0,2025-11-11 19:49:08


  **FRMAEWORK FOR PHASE 2 - SCRAPER.CSV** 
--------------
#### Phase Overview
In this phase, we ingest the raw Ecommerce Sales dataset and conduct a high-level inspection.
Goals:

- Load the dataset in a reproducible way.

- Examine basic metadata and structural integrity.

- Identify potential data quality issues for later cleaning.

- Document first impressions and possible red flags.

In [38]:
df_2 = pd.read_csv("../.data_cache/scraper.csv")
df_2.head()

,Name,Price,Ratings,Description Link,Category,Timestamp
0,"Blueing 15.6"" Laptop J4125 8GB+256GB SSD Stude...","₦ 234,000",3.9 out of 5,https://www.jumia.com.ng/customer/account/logi...,laptops,2025-11-13 01:35:13
1,Macbook PRO Laptop A1278 13.3 Inch Core I5 2.5...,"₦ 187,000",3.4 out of 5,https://www.jumia.com.ng/renewed-macbook-pro-l...,laptops,2025-11-13 01:35:13
2,"Blueing 14"" Laptop N3350 6GB+192GB SSD Portabl...","₦ 215,278",4.1 out of 5,https://www.jumia.com.ng/customer/account/logi...,laptops,2025-11-13 01:35:13
3,Hp EliteBook 840 G7 Intel Core I5 Touchscreen ...,"₦ 566,000",4 out of 5,https://www.jumia.com.ng/customer/account/logi...,laptops,2025-11-13 01:35:13
4,Hp Hp Hp EliteBook 840 G7 10th Gen Intel Core ...,"₦ 519,990",4.1 out of 5,https://www.jumia.com.ng/customer/account/logi...,laptops,2025-11-13 01:35:13


In [11]:
df_2.info()

<class 'pandas.DataFrame'>
RangeIndex: 166993 entries, 0 to 166992
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   Name              166993 non-null  str  
 1   Price             166993 non-null  str  
 2   Ratings           166993 non-null  str  
 3   Description Link  166993 non-null  str  
 4   Category          166993 non-null  str  
 5   Timestamp         166993 non-null  str  
dtypes: str(6)
memory usage: 7.6 MB


#### DATA STANDARDIZATION WORKFLOW -  
1. Create the Brand column

2. convert to appropriate data types

3. Standardize the Price column

4. Normalize the Rating column

**CREATING BRAND COLUMN**

In [39]:
df_2["Brand"] = df_2["Name"].str.split().str[0]
df_2["Brand"].value_counts()

Brand
Hp         52819
DELL       19706
Samsung    18317
Nokia      11088
Apple      10247
           ...  
Item           1
Fitbit         1
SJBOB          1
K9             1
Visita         1
Name: count, Length: 137, dtype: int64

In [40]:
df_2 = df_2.reindex(columns=['Brand', 'Name', 'Price', 'Ratings','Category','Description Link', 'Timestamp']) 
df_2.head() 

,Brand,Name,Price,Ratings,Category,Description Link,Timestamp
0,Blueing,"Blueing 15.6"" Laptop J4125 8GB+256GB SSD Stude...","₦ 234,000",3.9 out of 5,laptops,https://www.jumia.com.ng/customer/account/logi...,2025-11-13 01:35:13
1,Macbook,Macbook PRO Laptop A1278 13.3 Inch Core I5 2.5...,"₦ 187,000",3.4 out of 5,laptops,https://www.jumia.com.ng/renewed-macbook-pro-l...,2025-11-13 01:35:13
2,Blueing,"Blueing 14"" Laptop N3350 6GB+192GB SSD Portabl...","₦ 215,278",4.1 out of 5,laptops,https://www.jumia.com.ng/customer/account/logi...,2025-11-13 01:35:13
3,Hp,Hp EliteBook 840 G7 Intel Core I5 Touchscreen ...,"₦ 566,000",4 out of 5,laptops,https://www.jumia.com.ng/customer/account/logi...,2025-11-13 01:35:13
4,Hp,Hp Hp Hp EliteBook 840 G7 10th Gen Intel Core ...,"₦ 519,990",4.1 out of 5,laptops,https://www.jumia.com.ng/customer/account/logi...,2025-11-13 01:35:13


In [41]:
# Zero missing values in the dataset, which is a good sign for data quality.
df_2.isna().sum()

Brand               0
Name                0
Price               0
Ratings             0
Category            0
Description Link    0
Timestamp           0
dtype: int64

**CONVERSION OF DATA TYPES & STANDARDIZATION**  

In [ ]:
# Standardizing the ratings column by extracting the first numeric part  
df_2["Ratings"] = df_2["Ratings"].str.split().str[0]
if df_2["Ratings"] .isna().any() or (df_2["Ratings"] == "No").any():
    df_2["Ratings"] = df_2["Ratings"].fillna(0).replace("No", "No Ratings")

In [43]:
df_2["Ratings"]

0                3.9
1                3.4
2                4.1
3                  4
4                4.1
             ...    
166988             5
166989    No Ratings
166990    No Ratings
166991           4.3
166992    No Ratings
Name: Ratings, Length: 166993, dtype: object

In [44]:
# Cleaning the price column by removing the currency symbol and commas, then converting to numeric. 
# Also, converting the ratings to numeric.
df_2["Price"] = pd.to_numeric(df_2["Price"].str.replace("₦", "", regex=False).str.replace(",", "", regex=False), errors='coerce')
df_2["Ratings"] = pd.to_numeric(df_2["Ratings"], errors='coerce')  

In [45]:
df_2.head()

,Brand,Name,Price,Ratings,Category,Description Link,Timestamp
0,Blueing,"Blueing 15.6"" Laptop J4125 8GB+256GB SSD Stude...",234000.0,3.9,laptops,https://www.jumia.com.ng/customer/account/logi...,2025-11-13 01:35:13
1,Macbook,Macbook PRO Laptop A1278 13.3 Inch Core I5 2.5...,187000.0,3.4,laptops,https://www.jumia.com.ng/renewed-macbook-pro-l...,2025-11-13 01:35:13
2,Blueing,"Blueing 14"" Laptop N3350 6GB+192GB SSD Portabl...",215278.0,4.1,laptops,https://www.jumia.com.ng/customer/account/logi...,2025-11-13 01:35:13
3,Hp,Hp EliteBook 840 G7 Intel Core I5 Touchscreen ...,566000.0,4.0,laptops,https://www.jumia.com.ng/customer/account/logi...,2025-11-13 01:35:13
4,Hp,Hp Hp Hp EliteBook 840 G7 10th Gen Intel Core ...,519990.0,4.1,laptops,https://www.jumia.com.ng/customer/account/logi...,2025-11-13 01:35:13


  **FRMAEWORK FOR PHASE 3 - API_AUTH.CSV** 
--------------
#### Phase Overview
In this phase, we ingest the raw Ecommerce Sales dataset and conduct a high-level inspection.
Goals:

- Load the dataset in a reproducible way.

- Examine basic metadata and structural integrity.

- Identify potential data quality issues for later cleaning.

- Document first impressions and possible red flags.

In [46]:
df_3 = pd.read_csv("../.data_cache/api_auth.csv") 
df_3.head()

,City,Country,Latitude,Longitude,Timestamp_UTC,Temperature_C,Wind_KPH,Condition,AQI_US,CO,NO2,O3,PM2.5
0,Casablanca,Morocco,33.5931,-7.6164,1762477200,17.6,14.8,Partly Cloudy,1.0,159.85,17.75,50.0,12.95
1,Kampala,Uganda,0.3156,32.5656,1762477200,17.4,3.6,Patchy rain nearby,3.0,774.85,15.65,41.0,55.75
2,Lagos,Nigeria,6.4531,3.3958,1762479000,27.2,13.3,Clear,3.0,1673.85,39.55,18.0,42.85
3,Accra,Ghana,5.5500,-0.2167,1762479000,26.4,10.4,Partly Cloudy,1.0,412.85,11.05,32.0,6.65
4,Nairobi,Kenya,-1.2833,36.8167,1762479000,15.6,10.4,Clear,1.0,258.85,12.15,29.0,12.85


In [47]:
df_3.info()

<class 'pandas.DataFrame'>
RangeIndex: 827 entries, 0 to 826
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   City           827 non-null    str    
 1   Country        827 non-null    str    
 2   Latitude       827 non-null    float64
 3   Longitude      827 non-null    float64
 4   Timestamp_UTC  827 non-null    int64  
 5   Temperature_C  827 non-null    float64
 6   Wind_KPH       827 non-null    float64
 7   Condition      827 non-null    str    
 8   AQI_US         817 non-null    float64
 9   CO             817 non-null    float64
 10  NO2            817 non-null    float64
 11  O3             817 non-null    float64
 12  PM2.5          817 non-null    float64
dtypes: float64(9), int64(1), str(3)
memory usage: 84.1 KB


In [50]:
# Checking for missing values in the dataset to assess data quality and identify any potential issues that may need to be addressed before analysis.
df_3.isna().sum() 

City              0
Country           0
Latitude          0
Longitude         0
Timestamp_UTC     0
Temperature_C     0
Wind_KPH          0
Condition         0
AQI_US           10
CO               10
NO2              10
O3               10
PM2.5            10
dtype: int64

In [52]:
# Calculating the percentage of missing values in each column to better understand the extent of missing data.  
percent_missing = df_3.isna().sum() * 100 / len(df_3)
print(percent_missing)

City             0.00000
Country          0.00000
Latitude         0.00000
Longitude        0.00000
Timestamp_UTC    0.00000
Temperature_C    0.00000
Wind_KPH         0.00000
Condition        0.00000
AQI_US           1.20919
CO               1.20919
NO2              1.20919
O3               1.20919
PM2.5            1.20919
dtype: float64
